# TP13 — Pipeline bout-en-bout : parcours guidé

**Support à compléter ensemble, pas à pas — aucune solution préremplie.**

Objectif : orchestrer les briques existantes du brut aux exports ML, sans réécrire
le nettoyage du TP9 ni la conversion du TP12.

Énoncé de référence : [TP13](../../../docs/tps/tp-13-pipeline-bout-en-bout.md).
Documentation actuelle : [pipeline TP9](../../../docs/data/pipeline_tp9.md) et
[architecture des données](../../../docs/data/architecture.md).

Ce notebook est un espace de conception et de vérification. Le livrable final
sera une CLI utilisable sans Jupyter, des tests et une documentation de lancement.
Les fonctions proposées ci-dessous sont des squelettes, pas une architecture
imposée. Nous déciderons des interfaces avant de les compléter.

**Méthode :** lire une étape, proposer une réponse, écrire le minimum, vérifier,
puis seulement passer à la suivante. Ne pas compléter tous les TODO d’un coup.
Ne pas lancer de traitement complet ni de chargement PostgreSQL depuis ce support.
Les vérifications restent désactivées jusqu’à l’implémentation correspondante.

**Git :** branche `Klement/feat/tp13-pipeline`, sur le fork
`clement-welsch/NutriScope` uniquement. PR vers le `develop` du même fork,
sans fusion automatique ; aucune intervention sur le dépôt de Sacha.

Données : © Open Food Facts contributors —
[ODbL 1.0](https://opendatacommons.org/licenses/odbl/1-0/).

## 0 — Vérifier l’environnement et les points d’appui

Cellule fournie : bibliothèque standard uniquement, lecture de chemins, sans
réseau et sans écriture. Choisir un kernel Python 3.12 fonctionnel.
Le `.venv` du dépôt peut référencer un interpréteur absent : afficher le chemin
réel avant de conclure qu’un environnement fonctionne.

Lors de l’inspection du 12/11/2026, les 19 tests ciblés de `test_etl_pipeline.py`
et `test_storage.py` ont réussi. C’est un résultat de l’inspection, pas une
exécution de ce notebook ni une preuve de réussite du futur orchestrateur.

In [2]:
import sys
import json
from pathlib import Path

origine = Path.cwd().resolve()
RACINE = next((p for p in (origine, *origine.parents)
               if (p / "src/etl_pipeline.py").is_file()
               and (p / "docs/tps/tp-13-pipeline-bout-en-bout.md").is_file()), None)
if RACINE is None:
    raise RuntimeError("Ouvrir ce notebook depuis le dépôt NutriScope.")
print("Python :", sys.version.split()[0])
print("Interpréteur :", sys.executable)
print("Dépôt :", RACINE)
if str(RACINE) not in sys.path:
    sys.path.insert(0, str(RACINE))


Python : 3.12.10
Interpréteur : c:\Users\Administrateur\Documents\NutriScope\.venv\Scripts\python.exe
Dépôt : C:\Users\Administrateur\Documents\NutriScope


## 1 — Cartographier ce qui existe

| Besoin | Fichier / interface à lire | Point à examiner |
|---|---|---|
| Extraction France | `src/etl_pipeline.py` : `extraire_brut` | Source OFF Parquet avec champs imbriqués ; ce n’est pas un lecteur universel |
| Nettoyage | `src/cleaning.py` : `lire_brut`, `nettoyer` | Colonnes attendues, journal des règles et ordre des traitements |
| Rapport | `src/report.py` : `generer_rapport` | Avant, après, journal et destination |
| Tables relationnelles | `src/etl_pipeline.py` : `vers_tables_sql` | Les cinq tables viennent du même nettoyé |
| Catalogue plat | `src/storage/preparer_csv.py` : `preparer` | Rayon, sous-rayon, nom et marque |
| Conversion | `src/storage/convert_parquet.py` : `convertir` | Supprime/recrée sa destination ; pas encore de publication atomique |
| Export ML | `src/storage/export_features.py` : `exporter` | Date, refus d’écrasement, split et empreinte déjà implémentés |
| PostgreSQL | `src/load_database.py` : `main` | Relance la préparation, puis vide la base : ne pas l’appeler tel quel |

**TODO écrit :** dessiner le flux en cinq étapes et indiquer leurs entrées/sorties.
`charger_base` et `parquet` doivent lire exactement le même nettoyé.

**Question :** pourquoi appeler successivement `preparer_pipeline()` puis
`ecrire_csv_nettoye()` ferait-il répéter une partie du travail ?

**Validation :** expliquer où le nettoyage est fait une seule fois et où les
sorties relationnelles et analytiques se séparent. Aucun fichier à produire ici.

In [ ]:
# TODO : écrire ici le flux retenu et les contrats des cinq étapes.

# TODO : préciser les formats source réellement pris en charge en première version.
# TODO : identifier les adaptations nécessaires sans dupliquer les règles TP9.


## 2 — Premier exercice : construire le contrat de la CLI

Décision déjà prise ensemble : une CLI Python avec `argparse`.

```text
python -m nutriscope.pipeline <étape> --source ... --sortie ... --date AAAA-MM-JJ
```

- Étapes : `charger`, `nettoyer`, `charger_base`, `parquet`, `exports`, `all`.
- `--source` : fichier brut ; requis pour les étapes qui lisent la source.
- `--sortie` : racine des artefacts produits, pas un fichier individuel.
- `--date` : date du jeu à produire/rejouer ; distincte de l’heure réelle du lancement.
- Une option d’écrasement des exports sera ajoutée après définition de sa portée.

**TODO immédiat :** créer un `ArgumentParser` avec une description, rien de plus.
Montre ce premier essai avant d’ajouter les arguments.

Puis, après validation : déclarer le choix d’étape, les trois options et leurs aides.
Ne lancer aucun traitement pendant l’analyse des arguments.

**Questions :** `exports` a-t-il besoin de la source brute ? Une date invalide
doit-elle être refusée avant d’écrire des fichiers ?

**Cible de livraison :** `nutriscope/__init__.py` et `nutriscope/pipeline.py` à la
racine, avec la logique importable sans lancement automatique. La CLI finale
sera installable pour fonctionner également depuis un autre répertoire.

In [ ]:
import argparse


def construire_parseur():
    """Construire le parseur de la CLI sans déclencher de traitement."""
    # TODO 2.1 : créer le parseur et choisir une description claire.
    # TODO 2.2, ensuite : déclarer l’étape et les options convenues.
    # TODO 2.3, ensuite : retourner le parseur pour permettre les tests.
    raise NotImplementedError("Commencer par la création du parseur")


In [ ]:
VERIFIER_CLI = False
if VERIFIER_CLI:
    parseur = construire_parseur()
    args = parseur.parse_args([
        "charger", "--source", "extrait.parquet",
        "--sortie", "data/tp13_exercice", "--date", "2026-11-12",
    ])
    assert args.etape == "charger"
    assert Path(args.source).name == "extrait.parquet"
    assert Path(args.sortie).name == "tp13_exercice"
    assert str(args.date) == "2026-11-12"
    # TODO : vérifier --help, étape inconnue, option absente et date invalide.
    # La validation de date pourra vivre après le parsing : tester le contrat choisi.
    print("Cas nominal CLI vérifié ; compléter les cas d’erreur.")
else:
    print("Vérification CLI non exécutée : exercice à compléter.")


## 3 — Définir les chemins et le contexte d’une exécution

**TODO :** préciser une structure datée pour le brut, le nettoyé, les Parquets,
les exports, les manifestes et le journal. Conserver les noms attendus par les
consommateurs existants ou documenter leur adaptation.

La source par défaut se résout par rapport au dépôt/package ; `--source` et une
variable d’environnement peuvent la remplacer. Définir et documenter leur ordre
de priorité. Pour un chemin relatif fourni par l’utilisateur, choisir et expliquer
la base de résolution. Ne pas multiplier les chemins absolus propres au poste.

**TODO :** définir le contexte partagé par les étapes et une même forme de résultat
(étape, statut, nombre de lignes, fichiers produits, durée).
La date du jeu et l’horodatage du lancement ont deux rôles distincts.

**Vérifications à écrire :** changement de répertoire courant, chemin absolu,
priorité CLI/environnement/défaut, date valide/invalide et absence d’écriture
pendant la simple résolution des chemins.

In [ ]:
# TODO : proposer d’abord les champs du contexte et du résultat d’étape.
# TODO : définir la politique de résolution des chemins et la variable d’environnement.
# TODO : écrire une fonction de résolution sans lancer d’étape ni créer de dossier.
# TODO : tester ces choix dans un répertoire temporaire.


## 4 — Implémenter uniquement `charger`

**Avant de coder :** inspecter un petit échantillon du format retenu, les colonnes
et les types réels. La source OFF complète est un Parquet imbriqué ; un CSV plat
n’a pas le même contrat. Ne pas supposer que les deux se lisent de la même manière.

**TODO :** réutiliser l’extraction existante lorsqu’elle convient, conserver le
code en texte, écrire un brut daté et un manifeste avec source, empreinte,
volumétrie et licence. Ne pas nettoyer à cette étape.

**Point de contrôle :** petit jeu de test, zéros de tête préservés, schéma invalide
refusé clairement et source originale inchangée. Distinguer une fixture synthétique
pour les tests d’un extrait OFF réel pour la démonstration.

In [ ]:
def charger(contexte):
    """Produire le brut figé et son manifeste, sans nettoyage."""
    # TODO : valider source/format, déléguer à la brique adaptée, tracer les sorties.
    raise NotImplementedError("Implémenter l’étape charger après validation du contrat")

VERIFIER_CHARGER = False
if VERIFIER_CHARGER:
    # TODO : construire une petite fixture dans un dossier temporaire.
    # TODO : appeler charger et vérifier sorties, codes texte et manifeste.
    raise NotImplementedError("Ajouter les vérifications de charger")
else:
    print("Étape charger non exécutée.")


## 5 — Implémenter `nettoyer` et matérialiser la source commune

**TODO :** relire le brut produit par `charger`, appeler le nettoyage TP9 une seule
fois, puis conserver le nettoyé daté et son journal. Générer le rapport avant/après
à partir de ces mêmes données. Ne pas écrire par défaut sur le rapport TP9 versionné.

**Question :** sous quelle représentation conserver le nettoyé pour alimenter
`vers_tables_sql()` et `preparer()` sans perdre leurs colonnes attendues ?

**Vérifications :** cas de doublon, nutriment incohérent, manquant, comptages
avant/après, lien rapport/journal et mêmes codes pour les deux consommateurs.
Réutiliser les tests existants plutôt que recopier toutes leurs fixtures.

In [ ]:
def nettoyer_etape(contexte):
    """Produire le nettoyé commun, le rapport et le journal des règles."""
    # TODO : résoudre le brut validé puis réutiliser nettoyer() et generer_rapport().
    # TODO : persister le nettoyé avec les informations nécessaires aux deux branches.
    raise NotImplementedError("Implémenter l’étape nettoyer")

# TODO : écrire et exécuter les vérifications sur fixture temporaire.


## 6 — Brancher la sortie `parquet`

**TODO :** construire le catalogue plat depuis le nettoyé commun avec `preparer()`,
puis réutiliser `convertir()` sans lancer un deuxième nettoyage.

Le convertisseur actuel supprime son dossier cible. Concevoir un dossier temporaire
isolé, les contrôles de sortie et la publication du jeu complet avant d’autoriser
un remplacement. Un marqueur `_SUCCESS` est écrit en dernier et vérifié par les
consommateurs. Ne pas viser le catalogue actuel pendant les exercices.

**Vérifications :** comptage égal au nettoyé, types attendus, partitions complètes,
absence de partition orpheline après relance et absence de `_SUCCESS` après échec.
Une destination existante ne doit pas devenir une sortie valide partiellement écrite.

In [ ]:
def parquet(contexte):
    """Produire le catalogue partitionné depuis le nettoyé commun."""
    # TODO : préparer la table plate, convertir dans une destination isolée.
    # TODO : contrôler, publier, puis marquer la réussite selon la stratégie retenue.
    raise NotImplementedError("Implémenter l’étape parquet")

# TODO : tester les contrôles et un échec avant publication dans un dossier temporaire.


## 7 — Réutiliser `exports` et vérifier le split existant

`exporter()` fournit déjà date, refus d’écrasement sans `force`, split stratifié
par grade, graine 42, empreinte et schéma par export. Les tests existants vérifient
la stabilité produit par produit et celle des octets sur leur fixture.

**TODO :** résoudre un catalogue complet, vérifier son marqueur de réussite, puis
déléguer à `exporter()` avec les paramètres attendus. Définir la portée exacte
de `--force` : cette option ne doit jamais autoriser implicitement un vidage de base.

**Points de contrôle :** cinq grades sur une fixture assez grande, environ 20 %
de test par grade, partitions disjointes, même affectation par code après relance,
export protégé et schéma cohérent. La stabilité des octets dépend aussi des versions
logicielles : déclarer et comparer dans le même environnement.

L’export figé contient les règles d’usage : ajustements sur train uniquement,
validation croisée dans le train, test final réservé.

In [ ]:
def exports(contexte):
    """Exporter les features et leur split figé depuis un catalogue complet."""
    # TODO : résoudre l’entrée, contrôler _SUCCESS et appeler exporter().
    raise NotImplementedError("Brancher l’export existant")

# TODO : reprendre les assertions pertinentes de tests/test_storage.py.
# TODO : vérifier qu’un jeu inachevé est refusé avant l’export.


## 8 — Préparer `charger_base` sans toucher à la base actuelle

Le `main()` historique de `src/load_database.py` relance la préparation et vide
ensuite les tables. Il ne correspond pas encore au contrat d’une étape indépendante.

**TODO de conception :** séparer préparation, chargement et configuration ; utiliser
les cinq relations issues du même nettoyé que le Parquet ; valider les entrées avant
les opérations SQL. Éviter de recopier les insertions existantes.

**Première vérification :** tester le routage avec un chargeur simulé, explicitement
réservé au test. Aucun contact PostgreSQL requis pour cette vérification.

**Validation réelle ultérieure :** définir une base de test isolée et sa configuration.
Tout rechargement destructif de la base existante nécessitera une autorisation
explicite pour la cible exacte. Ne pas lancer `src/load_database.py` pour explorer.
Après validation sur la cible isolée : comparer les codes et volumes au nettoyé,
contrôler les clés et démontrer l’absence de doublons après relance.

In [ ]:
def charger_base(contexte):
    """Charger une cible PostgreSQL explicitement configurée depuis le nettoyé."""
    # TODO : concevoir le contrat avec le chargeur existant avant toute exécution SQL.
    raise NotImplementedError("Étape base à préparer sur une cible isolée")

# TODO : tester d’abord les appels et contrôles avec une dépendance simulée.
# Aucun appel réel à la base dans cette cellule.


## 9 — Relancer une étape, dater les sorties et tracer les erreurs

**TODO :** pour chaque étape, définir comment retrouver la sortie de la date demandée,
puis la dernière sortie complète compatible si la politique de repli l’autorise.
Ne pas mélanger silencieusement des versions de sources ; conserver la provenance.
En l’absence de prérequis, indiquer quelle étape lancer avant.

**TODO :** journaliser heure réelle, date du jeu, étape, durée (`perf_counter`),
statut, nombre de lignes et erreur éventuelle. Journaliser les échecs aussi.
Les horodatages variables restent dans les manifestes/journaux, pas dans les données.

**Vérifications :** exécution interrompue, prérequis absent, reprise isolée d’exports,
deux dates qui cohabitent, et deux exécutions identiques avec empreintes comparées.
Distinguer le refus volontaire d’écraser un export figé d’un défaut de relance.

In [ ]:
# TODO : écrire le résolveur d’artefacts datés et complets.
# TODO : décider du format du journal et des erreurs présentées par la CLI.
# TODO : tester succès, échec et reprise avec des fonctions d’étape simulées.
# TODO : vérifier que les journaux variables ne modifient pas les fichiers de données.


## 10 — Assembler `all` et extraire le point d’entrée

**TODO :** fixer l’ordre et la table de routage explicite des cinq étapes. `all`
s’arrête sur la première erreur et ne marque pas les suivantes comme réussies.
Il réutilise les mêmes fonctions que les commandes isolées.

**Avant un vrai parcours complet :** vérifier la cible de `charger_base` ; la présence
de `all` ne constitue pas une autorisation de vider la base de développement.
Un parcours sans base peut servir au développement mais doit être nommé et présenté
comme partiel, pas comme preuve de réussite des cinq étapes.

**TODO :** déplacer le code stable dans le package `nutriscope/`, ajouter le point
d’entrée protégé par `if __name__ == "__main__"`, les dépendances et le nécessaire
pour une installation reproductible. Le notebook importe ensuite le package ;
aucune deuxième implémentation de la CLI ne doit rester dans les cellules.

**Vérifications :** `--help` sans effet de bord, import sans exécution, étapes
isolées, ordre de `all`, arrêt après erreur et invocation depuis un autre dossier
une fois le package installé.

In [ ]:
# TODO : construire le routage et main() après validation des étapes isolées.
# TODO : tester l’ordre et l’arrêt sur erreur sans lancer de traitement coûteux.
# TODO : extraire vers nutriscope/pipeline.py, puis importer la version stable ici.


## 11 — Exécuter les vérifications puis mesurer sur des données réelles

Les tests rapides utilisent de petites fixtures temporaires ; les mesures utilisent
un extrait OFF réel identifié, puis éventuellement la source complète. Ne pas
annoncer une durée, une volumétrie ou une reproductibilité sans exécution observée.

| Vérification à préparer | Preuve attendue |
|---|---|
| Parsing et chemins | CLI invalide refusée, indépendance du répertoire courant |
| Cinq étapes et source commune | Contrats, identités et comptages cohérents |
| Idempotence | Deux exécutions, empreintes et affectations comparées |
| Reprise et erreur | Prérequis explicite, journal d’échec, pas de faux _SUCCESS |
| Export figé | Refus sans force, stabilité dans le même environnement |
| Chargement SQL | Cible isolée, intégrité, comptage égal au catalogue |
| Installation | Dépendances déclarées, nouvel environnement |

Viser au moins les douze tests pertinents demandés par le TP, sans confondre le
nombre total du dépôt et les contrats réellement couverts par le pipeline.

**Table de mesures à remplir après exécution :**

| Date du jeu / source / versions | Étape | Lignes | Durée | Statut | Artefacts / preuve |
|---|---|---:|---:|---|---|
| À renseigner | À renseigner | — | — | Non exécuté | — |

**Questions :** quelle étape domine la durée ? Les deux branches consomment-elles
la même version du nettoyé ? Que prouve une empreinte identique, et que ne prouve-t-elle pas ?

In [ ]:
# TODO : lancer les tests ciblés après extraction du code, puis relever les résultats.
# TODO : préparer un extrait réel et un espace de sorties isolé.
# TODO : mesurer uniquement les parcours effectivement exécutés et garder les journaux.
# TODO : ne lancer un rechargement SQL réel qu’après définition de la cible autorisée.


## 12 — Documenter, reproduire et préparer la PR sur le fork

**TODO :** rédiger `docs/data/pipeline.md` : flux réel, commandes, paramètres,
prérequis, politique de relance, sécurité de la cible SQL et durées mesurées.
Compléter le README avec création du venv, installation, données et sortie attendue.

Tester dans un clone frais de **ton fork** et un nouvel environnement, sans
contacter le dépôt de Sacha. Une reproduction locale dans un nouvel environnement
ne remplace pas la validation par une autre personne demandée par le TP : consigner
séparément qui a exécuté quoi et ce qui reste non vérifié.

**TODO :** journal Fait / Problème / Suite, commits cohérents et PR vers
`clement-welsch/NutriScope:develop`. Le tag `v0.3` n’est posé qu’après validation
réelle du jalon ; ne pas cocher les livrables par anticipation.

### Checklist de clôture

- [ ] Contrats et chemins définis, CLI installable.
- [ ] Cinq étapes indépendantes et `all` fonctionnels.
- [ ] Nettoyé commun à PostgreSQL et au Parquet.
- [ ] Sorties datées, manifestes, publication complète et _SUCCESS.
- [ ] Journal des succès et échecs avec durées observées.
- [ ] Export protégé et split existant vérifiés, sans recalcul côté TP14.
- [ ] Tests pertinents écrits et réellement exécutés.
- [ ] Parcours réel sur cible isolée et comptages vérifiés.
- [ ] Reproduction documentée ; validation externe distinguée de l’auto-test.
- [ ] Documentation et journal mis à jour.
- [ ] PR relue puis jalon/tag traités explicitement.

### Notes de séance

**Fait :** à compléter avec les actions réellement réalisées.

**Problème :** à compléter avec les erreurs et questions observées.

**Prochaine petite étape :** commencer par le parseur de la section 2.

Données : © Open Food Facts contributors — ODbL 1.0.